# HF SPECTER MWE

Dieses Notebook ist ein **MWE** (*Minimal Working Example*): das kleinstmoegliche, aber lauffaehige Beispiel fuer `InferenceClient.feature_extraction(...)` mit `allenai/specter`.

Es zeigt:
- Laden von `HF_TOKEN` aus `/home/ubuntu/Author_Name_Disambiguation/.env`
- Aufbau eines laengeren wissenschaftlichen Beispieltexts aus **Title + Abstract**
- Aufruf von `huggingface_hub.InferenceClient`
- Kurze Inspektion der Rueckgabeform

Das Repo nutzt produktiv denselben SPECTER-Vertrag (`Title [SEP] Abstract`), hat dort aber zusaetzlich Retry- und Shape-Normalisierung.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import numpy as np
from huggingface_hub import InferenceClient

REPO_ROOT = Path('/home/ubuntu/Author_Name_Disambiguation')
DOTENV_PATH = REPO_ROOT / '.env'


def load_env_value(env_path: Path, key: str) -> str:
    if not env_path.exists():
        raise FileNotFoundError(f'Missing env file: {env_path}')

    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('export '):
            line = line[len('export '):].strip()
        if '=' not in line:
            continue

        name, value = line.split('=', 1)
        if name.strip() != key:
            continue

        parsed = value.strip().strip("\"").strip("'")
        if not parsed:
            break
        os.environ[key] = parsed
        return parsed

    raise KeyError(f'{key} not found in {env_path}')


HF_TOKEN = os.environ.get('HF_TOKEN') or load_env_value(DOTENV_PATH, 'HF_TOKEN')
print(f'HF_TOKEN loaded from {DOTENV_PATH}: yes, length={len(HF_TOKEN)}')

HF_TOKEN loaded from /home/ubuntu/Author_Name_Disambiguation/.env: yes, length=37


In [2]:
title = 'Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings'
abstract = (
    'Author name disambiguation remains a critical bottleneck for the construction of reliable scholarly knowledge graphs, '
    'especially in large-scale bibliographic collections where homonyms, inconsistent initials, and affiliation drift are common. '
    'We present a graph-aware disambiguation pipeline that combines transformer-based document embeddings with citation-context signals, '
    'coauthor overlap, venue trajectories, and temporally constrained affiliation evidence. '
    'Our approach represents each publication as a fused document vector derived from title and abstract text, then augments pairwise scoring '
    'with neighborhood features extracted from local citation and collaboration graphs. '
    'In experiments on astronomy and computer-science benchmarks, the proposed method improves cluster purity and pairwise F1 over strong '
    'blocking-based baselines while remaining robust under sparse metadata conditions. '
    'A detailed ablation study shows that document-level semantic representations are particularly valuable when lexical name evidence is weak, '
    'and that graph cues reduce false merges among prolific authors publishing across adjacent subfields.'
)


def build_source_text(title: str, abstract: str) -> str:
    title = (title or '').strip()
    abstract = (abstract or '').strip()
    if title and abstract:
        return f'{title} [SEP] {abstract}'
    return title or abstract


paper_text = build_source_text(title, abstract)
print(title)
print()
print(f'Abstract characters: {len(abstract)}')
print(f'Combined request characters: {len(paper_text)}')
print()
print(paper_text[:500] + '...')

Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings

Abstract characters: 1136
Combined request characters: 1245

Graph-Aware Author Name Disambiguation with Citation Context and Transformer-Based Document Embeddings [SEP] Author name disambiguation remains a critical bottleneck for the construction of reliable scholarly knowledge graphs, especially in large-scale bibliographic collections where homonyms, inconsistent initials, and affiliation drift are common. We present a graph-aware disambiguation pipeline that combines transformer-based document embeddings with citation-context signals, coauthor overlap...


In [3]:
client = InferenceClient(
    provider='hf-inference',
    api_key=os.environ['HF_TOKEN'],
)

result = client.feature_extraction(
    paper_text,
    model='allenai/specter',
)

print(f'Response Python type: {type(result).__name__}')

Response Python type: ndarray


In [5]:
response_array = np.asarray(result, dtype=np.float32)

if response_array.ndim == 1:
    vector = response_array
    vector_strategy = 'already 1D'
elif response_array.ndim == 2:
    vector = response_array[0]
    vector_strategy = 'first token / CLS row'
elif response_array.ndim == 3:
    vector = response_array[0, 0]
    vector_strategy = 'batch[0], first token / CLS row'
else:
    raise ValueError(f'Unexpected response shape: {response_array.shape}')

summary = {
    'raw_shape': list(response_array.shape),
    'vector_shape': list(vector.shape),
    'vector_dtype': str(vector.dtype),
    'vector_strategy': vector_strategy,
    'vector_preview': vector[:10].round(6).tolist(),
    'vector_norm_l2': float(np.linalg.norm(vector)),
}

print('Note: hf-inference returns token-level features for this request; the repo uses the first token as the document embedding.')
print(json.dumps(summary, indent=2))
vector[:10]

Note: hf-inference returns token-level features for this request; the repo uses the first token as the document embedding.
{
  "raw_shape": [
    213,
    768
  ],
  "vector_shape": [
    768
  ],
  "vector_dtype": "float32",
  "vector_strategy": "first token / CLS row",
  "vector_preview": [
    -0.9996240139007568,
    1.1690900325775146,
    0.7262930274009705,
    -0.27566400170326233,
    0.7254520058631897,
    -0.7467550039291382,
    -0.13463500142097473,
    0.10361699759960175,
    0.26190999150276184,
    0.24876999855041504
  ],
  "vector_norm_l2": 22.541824340820312
}


array([-0.99962366,  1.1690905 ,  0.726293  , -0.2756641 ,  0.72545165,
       -0.7467554 , -0.13463487,  0.10361686,  0.26191002,  0.24877027],
      dtype=float32)

## Local Transformers MWE + CPU/GPU/API Showcase

Das offizielle SPECTER-Repo zeigt fuer die Hugging Face Transformers Library diesen lokalen Pfad:

```python
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
model = AutoModel.from_pretrained('allenai/specter')
inputs = tokenizer([title + tokenizer.sep_token + abstract], padding=True, truncation=True, return_tensors='pt', max_length=512)
result = model(**inputs)
embeddings = result.last_hidden_state[:, 0, :]
```

Hier erweitern wir das MWE auf drei Modi:
- `hf_api`: Remote via `huggingface_hub.InferenceClient`
- `local_cpu`: Lokal via `transformers` auf CPU
- `local_gpu`: Lokal via `transformers` auf GPU, falls CUDA verfuegbar ist

Der Laufzeitvergleich bleibt bewusst minimal: ein einzelnes Paper, eine einzelne Embedding-Anfrage, keine Batch-Optimierung.

In [6]:
from time import perf_counter

import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer

LOCAL_MODEL_NAME = 'allenai/specter'
CUDA_AVAILABLE = bool(torch.cuda.is_available())
CUDA_DEVICE_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None

runtime_env = {
    'torch_version': torch.__version__,
    'cuda_available': CUDA_AVAILABLE,
    'cuda_device_name': CUDA_DEVICE_NAME,
    'transformers_model': LOCAL_MODEL_NAME,
}

print(json.dumps(runtime_env, indent=2))

{
  "torch_version": "2.10.0+cu126",
  "cuda_available": true,
  "cuda_device_name": "NVIDIA A100-SXM4-80GB",
  "transformers_model": "allenai/specter"
}


In [7]:
def sync_device(device: str) -> None:
    if str(device).startswith('cuda') and torch.cuda.is_available():
        torch.cuda.synchronize()


def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    a = np.asarray(vec_a, dtype=np.float32)
    b = np.asarray(vec_b, dtype=np.float32)
    denom = float(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0.0:
        return float('nan')
    return float(np.dot(a, b) / denom)


def run_local_transformers_mwe(text: str, device: str) -> dict:
    started_total = perf_counter()
    started_load = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_NAME)
    model = AutoModel.from_pretrained(LOCAL_MODEL_NAME)
    model.to(device)
    model.eval()
    sync_device(device)
    load_seconds = perf_counter() - started_load

    inputs = tokenizer(
        [text],
        padding=True,
        truncation=True,
        return_tensors='pt',
        max_length=512,
    )
    inputs = {key: value.to(device) for key, value in inputs.items()}

    with torch.no_grad():
        sync_device(device)
        started_first = perf_counter()
        outputs = model(**inputs)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        sync_device(device)
        first_inference_seconds = perf_counter() - started_first

        sync_device(device)
        started_warm = perf_counter()
        warm_outputs = model(**inputs)
        warm_cls_vector = warm_outputs.last_hidden_state[:, 0, :]
        sync_device(device)
        warm_inference_seconds = perf_counter() - started_warm

    vector = warm_cls_vector[0].detach().cpu().numpy().astype(np.float32, copy=False)
    raw_shape = list(warm_outputs.last_hidden_state.shape)

    return {
        'mode': f'local_{device}',
        'backend': 'transformers',
        'device': device,
        'available': True,
        'load_seconds': float(load_seconds),
        'first_inference_seconds': float(first_inference_seconds),
        'warm_inference_seconds': float(warm_inference_seconds),
        'end_to_end_seconds': float(perf_counter() - started_total),
        'raw_shape': raw_shape,
        'vector_shape': list(vector.shape),
        'vector_norm_l2': float(np.linalg.norm(vector)),
        'vector': vector,
    }


def run_hf_api_mwe(text: str) -> dict:
    started = perf_counter()
    api_client = InferenceClient(provider='hf-inference', api_key=os.environ['HF_TOKEN'])
    api_result = api_client.feature_extraction(text, model=LOCAL_MODEL_NAME)
    response_array = np.asarray(api_result, dtype=np.float32)

    if response_array.ndim == 1:
        vector = response_array
    elif response_array.ndim == 2:
        vector = response_array[0]
    elif response_array.ndim == 3:
        vector = response_array[0, 0]
    else:
        raise ValueError(f'Unexpected HF API response shape: {response_array.shape}')

    return {
        'mode': 'hf_api',
        'backend': 'huggingface_hub',
        'device': 'remote',
        'available': True,
        'load_seconds': 0.0,
        'first_inference_seconds': float(perf_counter() - started),
        'warm_inference_seconds': float('nan'),
        'end_to_end_seconds': float(perf_counter() - started),
        'raw_shape': list(response_array.shape),
        'vector_shape': list(vector.shape),
        'vector_norm_l2': float(np.linalg.norm(vector)),
        'vector': vector.astype(np.float32, copy=False),
    }

In [8]:
benchmark_runs: list[dict] = []

benchmark_runs.append(run_hf_api_mwe(paper_text))
benchmark_runs.append(run_local_transformers_mwe(paper_text, device='cpu'))

if CUDA_AVAILABLE:
    benchmark_runs.append(run_local_transformers_mwe(paper_text, device='cuda'))
else:
    benchmark_runs.append({
        'mode': 'local_gpu',
        'backend': 'transformers',
        'device': 'cuda',
        'available': False,
        'load_seconds': float('nan'),
        'first_inference_seconds': float('nan'),
        'warm_inference_seconds': float('nan'),
        'end_to_end_seconds': float('nan'),
        'raw_shape': None,
        'vector_shape': None,
        'vector_norm_l2': float('nan'),
        'vector': None,
    })

runtime_table = pd.DataFrame([
    {
        key: value
        for key, value in row.items()
        if key != 'vector'
    }
    for row in benchmark_runs
])

runtime_table[['mode', 'backend', 'device', 'available', 'load_seconds', 'first_inference_seconds', 'warm_inference_seconds', 'end_to_end_seconds', 'raw_shape', 'vector_shape', 'vector_norm_l2']]

2026-03-31 14:31:27.811598: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


,mode,backend,device,available,load_seconds,first_inference_seconds,warm_inference_seconds,end_to_end_seconds,raw_shape,vector_shape,vector_norm_l2
0,hf_api,huggingface_hub,remote,True,0.000000,4.846644,NaN,4.846651,"[213, 768]",[768],22.541824
1,local_cpu,transformers,cpu,True,4.694939,0.133180,0.126028,4.956930,"[1, 213, 768]",[768],22.541826
2,local_cuda,transformers,cuda,True,1.581310,2.055001,0.009952,3.648467,"[1, 213, 768]",[768],22.541824


In [9]:
available_vectors = {
    row['mode']: row['vector']
    for row in benchmark_runs
    if row.get('available') and row.get('vector') is not None
}

comparison_rows = []
reference_modes = list(available_vectors)
for left_mode in reference_modes:
    for right_mode in reference_modes:
        if left_mode >= right_mode:
            continue
        comparison_rows.append({
            'left_mode': left_mode,
            'right_mode': right_mode,
            'cosine_similarity': cosine_similarity(available_vectors[left_mode], available_vectors[right_mode]),
        })

comparison_table = pd.DataFrame(comparison_rows)
print('Interpretation: local CPU/GPU use the official transformers path from the SPECTER repo; hf_api uses the remote inference endpoint.')
display(runtime_table)
display(comparison_table)

for row in benchmark_runs:
    preview = None if row['vector'] is None else np.asarray(row['vector'])[:8].round(6).tolist()
    print(f"{row['mode']}: preview={preview}")

Interpretation: local CPU/GPU use the official transformers path from the SPECTER repo; hf_api uses the remote inference endpoint.


,mode,backend,device,available,load_seconds,first_inference_seconds,warm_inference_seconds,end_to_end_seconds,raw_shape,vector_shape,vector_norm_l2
0,hf_api,huggingface_hub,remote,True,0.000000,4.846644,NaN,4.846651,"[213, 768]",[768],22.541824
1,local_cpu,transformers,cpu,True,4.694939,0.133180,0.126028,4.956930,"[1, 213, 768]",[768],22.541826
2,local_cuda,transformers,cuda,True,1.581310,2.055001,0.009952,3.648467,"[1, 213, 768]",[768],22.541824


,left_mode,right_mode,cosine_similarity
0,hf_api,local_cpu,1.0
1,hf_api,local_cuda,1.0
2,local_cpu,local_cuda,1.0


hf_api: preview=[-0.9996240139007568, 1.1690900325775146, 0.7262930274009705, -0.27566400170326233, 0.7254520058631897, -0.7467550039291382, -0.13463500142097473, 0.10361699759960175]
local_cpu: preview=[-0.9996230006217957, 1.169090986251831, 0.7262930274009705, -0.27566400170326233, 0.7254520058631897, -0.7467560172080994, -0.13463400304317474, 0.10361699759960175]
local_cuda: preview=[-0.9996240139007568, 1.169090986251831, 0.7262920141220093, -0.27566298842430115, 0.7254520058631897, -0.7467560172080994, -0.13463400304317474, 0.10361699759960175]


In [10]:
minimal_runtime_compare = runtime_table.copy()
minimal_runtime_compare['mwe_runtime_seconds'] = minimal_runtime_compare.apply(
    lambda row: row['warm_inference_seconds'] if row['backend'] == 'transformers' else row['first_inference_seconds'],
    axis=1,
)
minimal_runtime_compare = minimal_runtime_compare[
    ['mode', 'backend', 'device', 'mwe_runtime_seconds', 'first_inference_seconds', 'warm_inference_seconds']
].sort_values('mwe_runtime_seconds', na_position='last').reset_index(drop=True)

print('Minimal runtime comparison: local transformers uses warm inference after model load; hf_api uses request time.')
display(minimal_runtime_compare)

Minimal runtime comparison: local transformers uses warm inference after model load; hf_api uses request time.


,mode,backend,device,mwe_runtime_seconds,first_inference_seconds,warm_inference_seconds
0,local_cuda,transformers,cuda,0.009952,2.055001,0.009952
1,local_cpu,transformers,cpu,0.126028,0.133180,0.126028
2,hf_api,huggingface_hub,remote,4.846644,4.846644,NaN
